# Hurdle Regression Analysis of Hazardous Liquid Pipeline Shutdowns

Code accompanying:

> Ramezanpour, E. & Hainen, A. *Statistical Modeling of the Probability and Duration of Hazardous Liquid Pipeline Shutdowns: A Hurdle Regression Approach.*  Infrastructures 2026, 11, 177. https://doi.org/10.3390/infrastructures11050177

## What this notebook does

A two-stage hurdle model on PHMSA hazardous liquid pipeline incident data (2010–2025):

- **Stage 1** — logistic regression for the probability of a shutdown given an incident, using **pre-decision predictors only** (variables observable at the time the operator decides whether to shut down).
- **Stage 2** — lognormal OLS for the duration of a shutdown, conditional on a shutdown occurring; uses the full predictor pool.

## Data

Place the PHMSA file at `./data/accident_hazardous_liquid_jan2010_present.txt` (tab-separated, ISO-8859-1).

Download from: https://www.phmsa.dot.gov/data-and-statistics/pipeline/pipeline-incident-20-year-trends

Also expected in `./data/`:
- `Feature name Dictionary.csv` — column → human-readable description
- `numeric_column_summary.csv` — column-level missingness + drop flag for numerics
- `categorical_uner80null_uniqueValues_imputation.csv` — same for categoricals

The full archive is also available on FigShare: https://doi.org/10.6084/m9.figshare.30688115

## Notebook structure

| Section | Produces |
|---|---|
| 1–4 | Data loading, response construction, variable taxonomy, model frames |
| 5 | Stage 1 fit (LASSO → forward selection → cleanup) |
| 6 | Stage 2 fit (same procedure) |
| 7 | Tables 2 (categorical descriptives) and 3 (numerical descriptives) |
| 8 | Table 5 (hurdle coefficients) and Table 4 (model fit) |
| 9 | Figures 1, 2, 3, 4 |
| 10 | Supplementary robustness checks (Tables S1–S5, Figure S1) |
| 11 | Saves model artifacts |

## Runtime

The MICE blocks in Section 10 (BayesianRidge and PMM, m=10 each) take several minutes. Everything else runs in well under a minute.

## 0. Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import font_manager
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import scipy.stats as stats
from scipy.stats import chi2, norm

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegressionCV, LassoCV, BayesianRidge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, accuracy_score, roc_curve,
                              precision_score, recall_score, f1_score)

# ----- Font setup (Times New Roman for paper figures) -----
# Tries to locate the font; falls back to whatever matplotlib has if not found.
def _resolve_tnr():
    candidates = [
        r'C:\Windows\Fonts\times.ttf',
        '/Library/Fonts/Times New Roman.ttf',
        '/System/Library/Fonts/Supplemental/Times New Roman.ttf',
        '/usr/share/fonts/truetype/msttcorefonts/Times_New_Roman.ttf',
    ]
    for p in candidates:
        if os.path.exists(p):
            font_manager.fontManager.addfont(p)
            return font_manager.FontProperties(fname=p)
    # Fall back — matplotlib will use the next available serif
    return font_manager.FontProperties(family='serif')

tnr = _resolve_tnr()

plt.rcParams['font.family']      = 'serif'
plt.rcParams['font.serif']       = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['pdf.fonttype']     = 42   # embed TrueType in PDF
plt.rcParams['ps.fonttype']      = 42
plt.rcParams['figure.dpi']       = 100
plt.rcParams['savefig.dpi']      = 600

plt.rcParams['font.size']        = 11
plt.rcParams['axes.labelsize']   = 11
plt.rcParams['axes.titlesize']   = 12
plt.rcParams['xtick.labelsize']  = 10
plt.rcParams['ytick.labelsize']  = 10
plt.rcParams['legend.fontsize']  = 10
plt.rcParams['figure.titlesize'] = 12

sns.set_style('whitegrid')

# ----- Helpers used across plots -----
def apply_tnr(ax):
    """Force Times New Roman on every text element of an axis."""
    ax.xaxis.label.set_fontproperties(tnr)
    ax.yaxis.label.set_fontproperties(tnr)
    if ax.get_title():
        ax.title.set_fontproperties(tnr)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_fontproperties(tnr)
    leg = ax.get_legend()
    if leg is not None:
        for t in leg.get_texts():
            t.set_fontproperties(tnr)

def comma_if_5plus(x, pos):
    """Tick formatter: add commas only for numbers with 5+ digits."""
    if abs(x) >= 10000:
        return f'{x:,.0f}'
    return f'{x:.0f}'

RNG = 42

In [ ]:
# Paths — relative to this notebook
DATA_PATH = os.path.join(os.path.dirname(os.path.abspath('.')), 'data')
OUT_PATH  = os.path.join(os.path.dirname(os.path.abspath('.')), 'output')

# If you keep the standard layout (notebook in repo root, data/ and output/ alongside):
if not os.path.isdir(DATA_PATH):
    DATA_PATH = './data'
    OUT_PATH  = './output'

os.makedirs(OUT_PATH, exist_ok=True)
print(f'DATA_PATH = {os.path.abspath(DATA_PATH)}')
print(f'OUT_PATH  = {os.path.abspath(OUT_PATH)}')

## 1. Load Data

Three sources:
- PHMSA hazardous liquid incident dataset (2010–2025)
- Feature dictionary (column-name → human-readable description)
- Numeric and categorical column-summary tables (used to filter predictors by missingness)


In [ ]:
# PHMSA incident data
df = pd.read_csv(
    os.path.join(DATA_PATH, 'accident_hazardous_liquid_jan2010_present.txt'),
    sep='\t', encoding='ISO-8859-1', low_memory=False,
)
print(f'Raw PHMSA shape: {df.shape}')

# Feature description map (used in final tables)
feature_map = pd.read_csv(
    os.path.join(DATA_PATH, 'Feature name Dictionary.csv'),
    encoding='ISO-8859-1', low_memory=False,
)


## 2. Build the Response Variable

The response is `Shutdown_Duration_Hours` — a hurdle variable:
- **0** if the incident did not result in a shutdown (`SHUTDOWN_DUE_ACCIDENT_IND == 'NO'`)
- **>0** continuous hours if it did

Cleaning steps:
1. Drop rows with missing `SHUTDOWN_DUE_ACCIDENT_IND`
2. For `YES` rows missing both shutdown/restart timestamps — drop (no way to compute duration)
3. Compute duration in hours from `RESTART_DATETIME - SHUTDOWN_DATETIME`
4. Drop zero-duration rows (data-entry artifact)


In [ ]:
# Drop rows with missing shutdown indicator
data = df.dropna(subset=['SHUTDOWN_DUE_ACCIDENT_IND']).copy()
print(f"After dropping missing shutdown indicator: {len(data)}")

# Drop YES rows with no shutdown/restart timestamps (can't compute duration)
bad = data[
    (data['SHUTDOWN_DUE_ACCIDENT_IND'] == 'YES')
    & data['SHUTDOWN_DATETIME'].isna()
    & data['RESTART_DATETIME'].isna()
]
data = data.drop(bad.index)
print(f"Dropped {len(bad)} YES rows with no timestamps  →  {len(data)}")

# Compute Shutdown_Duration_Hours
data['SHUTDOWN_DATETIME'] = pd.to_datetime(data['SHUTDOWN_DATETIME'])
data['RESTART_DATETIME']  = pd.to_datetime(data['RESTART_DATETIME'])

mask = data['SHUTDOWN_DUE_ACCIDENT_IND'] == 'YES'
data['Shutdown_Duration_Hours'] = np.nan
data.loc[mask, 'Shutdown_Duration_Hours'] = (
    (data.loc[mask, 'RESTART_DATETIME'] - data.loc[mask, 'SHUTDOWN_DATETIME'])
    .dt.total_seconds() / 3600
)

# Drop zero-duration rows and any YES rows still missing timestamps
data = data.drop(data[data['Shutdown_Duration_Hours'] == 0].index)
data = data.drop(data[
    (data['SHUTDOWN_DUE_ACCIDENT_IND'] == 'YES')
    & (data['SHUTDOWN_DATETIME'].isna() | data['RESTART_DATETIME'].isna())
].index)

print(f"\nFinal shape: {data.shape}")
print(f"Shutdown rate: {(data['SHUTDOWN_DUE_ACCIDENT_IND']=='YES').mean():.3f}")
print(f"\nDuration summary (hours, shutdowns only):")
print(data.loc[mask, 'Shutdown_Duration_Hours'].describe().round(2))


## 3. Predictor Selection and Pre-processing

### 3.1 Variable taxonomy

Each candidate predictor is classified as **pre-decision** (information available at the time the operator decides whether to shut down) or **post-decision** (a consequence or outcome of the incident, observed only after the shutdown decision).

The Stage 1 (logistic) model uses pre-decision variables only — this avoids the temporal-ordering and post-treatment-bias concerns that arise when post-incident outcomes are used to predict the shutdown decision.

Stage 2 (duration) is conditional on shutdown having occurred, so post-decision variables are admissible there.

In [ ]:
PREDECISION_VARS = {
    # ---- Pre-decision: pipeline / infrastructure / operating context ----
    'PIPE_FACILITY_TYPE':       'pre',
    'PIPELINE_FUNCTION':        'pre',
    'MATERIAL_INVOLVED':        'pre',
    'SCADA_IN_PLACE_IND':       'pre',
    'CPM_IN_PLACE_IND':         'pre',
    'FEDERAL':                  'pre',
    'INCIDENT_AREA_TYPE':       'pre',
    'LOCATION_TYPE':            'pre',
    'SYSTEM_PART_INVOLVED':     'pre',
    'CROSSING':                 'pre',
    'COULD_BE_HCA':             'pre',
    'COMMODITY_RELEASED_TYPE':  'pre',
    'CAUSE':                    'pre',
    'RELEASE_TYPE':             'pre',
    'ACCIDENT_PSIG':            'pre',
    'MOP_PSIG':                 'pre',
    'ACCIDENT_PRESSURE':        'pre',
    'PRESSURE_RESTRICTION_IND': 'pre',
    'reaction_delay_ind':       'pre',
    'reaction_time(h)':         'pre',

    # ---- Post-decision: consequences / outcomes ----
    'PRPTY':                      'post',
    'UNINTENTIONAL_RELEASE_BBLS': 'post',
    'RECOVERED_BBLS':             'post',
    'NUM_PUB_EVACUATED':          'post',
    'COMMODITY_REACHED_HCA':      'post',
    'WATER_CONTAM_IND':           'post',
    'SOIL_CONTAMINATION':         'post',
    'WILDLIFE_IMPACT_IND':        'post',
    'REMEDIATION_IND':            'post',
    'LONG_TERM_ASSESSMENT':       'post',
    'INJURY_IND':                 'post',
    'FATALITY_IND':               'post',
    'IGNITE_IND':                 'post',
    'EXPLODE_IND':                'post',  # broken in PHMSA reporting after 2021 — drop entirely below
}

PRE_BASES  = [k for k, v in PREDECISION_VARS.items() if v == 'pre']
POST_BASES = [k for k, v in PREDECISION_VARS.items() if v == 'post']
print(f"Pre-decision base vars:  {len(PRE_BASES)}")
print(f"Post-decision base vars: {len(POST_BASES)}")


### 3.2 Numerical predictor list

Filter from `numeric_column_summary.csv`:
- `Missing_Percentage < 20%`
- `Drop?` flag not set
- Drop FATAL/INJURE counts (highly skewed, mostly zero, weak signal)

Engineer `reaction_time(h)` = `ON_SITE_DATETIME - INCIDENT_IDENTIFIED_DATETIME`.


In [ ]:
num_summary = pd.read_csv(
    os.path.join(DATA_PATH, 'numeric_column_summary.csv'),
    encoding='ISO-8859-1',
).fillna(0)

numeric_columns_list = num_summary.loc[
    (num_summary['Missing_Percentage'] < 20) & (num_summary['Drop?'] != 1),
    'columns',
].tolist()

for c in ['FATAL', 'INJURE']:
    if c in numeric_columns_list:
        numeric_columns_list.remove(c)

# Engineer reaction_time
data['Reaction_time'] = (
    pd.to_datetime(data['ON_SITE_DATETIME'])
    - pd.to_datetime(data['INCIDENT_IDENTIFIED_DATETIME'])
).dt.total_seconds() / 3600
data.loc[data['Reaction_time'] < 0, 'Reaction_time'] = np.nan
data['reaction_time(h)'] = data['Reaction_time']

numeric_columns_list.append('reaction_time(h)')

print(f"Numerical predictors ({len(numeric_columns_list)}):")
for c in numeric_columns_list: print(f"  {c}")


### 3.3 Categorical predictor list

Same filter. Two simplifications applied at the source:
- `COMMODITY_RELEASED_TYPE` and `SYSTEM_PART_INVOLVED` collapse rare categories into `OTHER`
- `EXPLODE_IND` dropped entirely (broken in PHMSA reporting after 2021)

Engineer a binary `reaction_delay_ind` from the response time.


In [ ]:
cat_summary = pd.read_csv(
    os.path.join(DATA_PATH, 'categorical_uner80null_uniqueValues_imputation.csv'),
    encoding='ISO-8859-1',
)

categorical_columns_list = cat_summary.loc[
    (cat_summary['drop?'] != 1) & (cat_summary['missing_percentage'] < 20),
    'Column',
].tolist()

# Drop fields not used as predictors
for c in ['ACCIDENT_IDENTIFIER', 'EXPLODE_IND']:
    if c in categorical_columns_list:
        categorical_columns_list.remove(c)
        print(f"Removed {c} from categorical list")

# Simplify COMMODITY_RELEASED_TYPE and SYSTEM_PART_INVOLVED
def simplify_commodity(x):
    keep = {
        'CRUDE OIL',
        'REFINED AND/OR PETROLEUM PRODUCT (NON-HVL) WHICH IS A LIQUID AT AMBIENT CONDITIONS',
        'HVL OR OTHER FLAMMABLE OR TOXIC FLUID WHICH IS A GAS AT AMBIENT CONDITIONS',
    }
    return x if x in keep else 'OTHER'

def simplify_system_part(x):
    keep = {
        'ONSHORE PIPELINE, INCLUDING VALVE SITES',
        'ONSHORE PUMP/METER STATION EQUIPMENT AND PIPING',
        'ONSHORE TERMINAL/TANK FARM EQUIPMENT AND PIPING',
    }
    return x if x in keep else 'OTHER'

data['COMMODITY_RELEASED_TYPE'] = data['COMMODITY_RELEASED_TYPE'].apply(simplify_commodity)
data['SYSTEM_PART_INVOLVED']    = data['SYSTEM_PART_INVOLVED'].apply(simplify_system_part)

# Engineer reaction_delay_ind
data['reaction_delay_ind'] = data['Reaction_time'].apply(
    lambda x: 1 if pd.notna(x) and x > 0 else (0 if pd.notna(x) else pd.NA)
)
categorical_columns_list.append('reaction_delay_ind')

print(f"\nCategorical predictors ({len(categorical_columns_list)}):")
for c in categorical_columns_list: print(f"  {c}")


## 4. Build Model Frames

1. Complete-case index over predictors + response
2. **No trimming** explicitly requested keeping the full distribution. Trimmed sensitivity reported in Section 10.4.
3. One-hot encode categoricals (`drop_first=True` to avoid perfect multicollinearity)
4. `log1p` transform numericals (heavy right skew on PRPTY, UNINTENTIONAL_RELEASE_BBLS, etc.)
5. Stage 1 frame: full sample. Stage 2 frame: shutdowns only.


In [ ]:
data_categorical = data[categorical_columns_list].copy()
data_numerical   = data[numeric_columns_list].copy()

# Complete-case index
clean_idx = pd.concat([data_categorical, data_numerical], axis=1).dropna().index
duration_clean = data['Shutdown_Duration_Hours'].loc[clean_idx].dropna()

analytical_idx = clean_idx   # untrimmed — keep all complete cases

print(f"Complete-case sample (untrimmed): {len(analytical_idx)}")

# Encode + log1p
data_num_log     = np.log1p(data_numerical.loc[analytical_idx])
data_cat_encoded = pd.get_dummies(
    data_categorical.loc[analytical_idx], drop_first=True
).astype(int)
df_merged = pd.concat([data_cat_encoded, data_num_log], axis=1).dropna()

# Stage 1 frame (all incidents)
X_log = df_merged.drop(columns='SHUTDOWN_DUE_ACCIDENT_IND_YES')
y_log = df_merged['SHUTDOWN_DUE_ACCIDENT_IND_YES']

# Stage 2 frame (shutdowns only)
shutdown_mask = y_log == 1
X_reg = X_log.loc[shutdown_mask]
y_reg = np.log(duration_clean).loc[X_reg.index]
y_reg_raw = np.exp(y_reg)

print(f"\nStage 1: n = {len(X_log)}, p_candidates = {X_log.shape[1]}")
print(f"Stage 2: n = {len(X_reg)}, shutdown rate = {y_log.mean():.3f}")


## 5. Stage 1 — Pre-decision Logistic Regression

Models `P(shutdown | incident)` using **only pre-decision variables**.

Selection procedure:
1. **LASSO** (L1-penalized logistic) on standardized features → ranks by |coefficient|
2. **Forward selection** in LASSO-ranked order, retaining only features with `p < 0.05`; if a newly added feature renders an existing one non-significant, the older one is dropped
3. **Secondary pass** — checks if any unselected pool features become significant given the current core
4. **Final cleanup** — joint refit, drop any remaining non-significant features


### 5.1 Build the pre-decision pool

In [ ]:
# Build the pre-decision feature pool.
# Requires X_log to exist (built in Section 4 — if it doesn't, run that first).
assert 'X_log' in dir(), "X_log is missing — re-run Section 4 (Build Model Frames) first."

# Match every X_log column that traces back to a pre-decision base var
predecision_pool = []
for col in X_log.columns:
    if any(col == b or col.startswith(b + '_') for b in PRE_BASES):
        predecision_pool.append(col)

# Sanity check — no post-decision columns leaked in
leaks = [c for c in predecision_pool
         if any(c == b or c.startswith(b + '_') for b in POST_BASES)]
assert not leaks, f"Post-decision leak: {leaks}"

print(f"Pre-decision pool: {len(predecision_pool)} features")

### 5.2 LASSO ranking

In [ ]:
X_pool = X_log[predecision_pool].copy()
X_std  = pd.DataFrame(
    StandardScaler().fit_transform(X_pool),
    columns=X_pool.columns, index=X_pool.index,
)

lasso = LogisticRegressionCV(
    Cs=20, cv=5, penalty='l1', solver='saga',
    scoring='neg_log_loss', max_iter=10000,
    random_state=RNG, n_jobs=-1,
).fit(X_std, y_log)

lasso_coefs   = pd.Series(lasso.coef_.ravel(), index=X_pool.columns)
lasso_ranking = lasso_coefs.abs().sort_values(ascending=False)

print(f"LASSO optimal C = {lasso.C_[0]:.4f}  (lambda ≈ {1/lasso.C_[0]:.4f})")
print(f"Non-zero LASSO coefficients: {(lasso_ranking > 0).sum()} / {len(lasso_ranking)}\n")
print("Top 15 by |LASSO coefficient|:")
print(lasso_ranking.head(15).round(4).to_string())


### 5.3 Forward selection (with monitoring)

In [ ]:
# ============================================================
# Stage 1 — Forward selection with monitoring
# ============================================================
print("="*70)
print("FORWARD PASS (LASSO-ranked order)")
print("="*70)

selected = []
forward_drops = []

for col in lasso_ranking.index:
    selected.append(col)
    res = sm.Logit(y_log, sm.add_constant(X_log[selected])).fit(disp=False)
    p_new = res.pvalues[col]

    if p_new >= 0.05:
        selected.remove(col)
        print(f"  ❌ {col:<70} p={p_new:.3f}")
        continue

    print(f"  ✅ {col:<70} p={p_new:.3f}")

    drops = [(c, res.pvalues[c]) for c in selected
             if c != col and res.pvalues[c] >= 0.05]
    for d, p_d in drops:
        print(f"     ⚠️  → '{d}' became insignificant (p={p_d:.3f}), dropping")
        forward_drops.append((col, d, p_d))
        selected.remove(d)

print(f"\n→ {len(selected)} features kept after forward pass")

# ----- SECONDARY PASS -----
print("\n" + "="*70)
print("SECONDARY PASS")
print("="*70)

core = selected.copy()
secondary_added, secondary_rejected = [], []

for f in [x for x in predecision_pool if x not in core]:
    trial = core + [f]
    res = sm.Logit(y_log, sm.add_constant(X_log[trial])).fit(disp=False)
    p_new = res.pvalues.get(f, 1.0)
    if p_new >= 0.05:
        continue
    harmed = [(c, res.pvalues.get(c, 1.0)) for c in core
              if res.pvalues.get(c, 1.0) >= 0.05]
    if not harmed:
        core.append(f)
        secondary_added.append((f, p_new))
        print(f"  ✅ {f:<70} p={p_new:.3f}  (added)")
    else:
        harmed_str = ", ".join(f"{c}(p={p:.3f})" for c, p in harmed)
        secondary_rejected.append((f, p_new, harmed))
        print(f"  ⚠️  {f:<70} p={p_new:.3f}  REJECTED — would harm: {harmed_str}")

if not secondary_added and not secondary_rejected:
    print("  (none — no pool features become significant on top of core)")

# ----- FINAL CLEANUP -----
print("\n" + "="*70)
print("FINAL CLEANUP")
print("="*70)

final_features_s1 = core
for it in range(5):
    res = sm.Logit(y_log, sm.add_constant(X_log[final_features_s1])).fit(disp=False)
    nonsig = [(f, res.pvalues.get(f, 1.0)) for f in final_features_s1
              if res.pvalues.get(f, 1.0) >= 0.05]
    if not nonsig:
        if it == 0:
            print("  (no cleanup needed — all features significant)")
        break
    print(f"  Iteration {it+1}: dropping")
    for f, p in nonsig:
        print(f"     • {f}  p={p:.3f}")
    final_features_s1 = [f for f in final_features_s1 if f not in {n[0] for n in nonsig}]

# ----- SUMMARY -----
print("\n" + "="*70)
print("STAGE 1 SELECTION SUMMARY")
print("="*70)
print(f"Final features: {len(final_features_s1)}")
for f in final_features_s1:
    print(f"  • {f}")

if forward_drops:
    print(f"\nForward-pass interaction drops ({len(forward_drops)}):")
    for added, dropped, p in forward_drops:
        print(f"  '{added}' bumped out '{dropped}' (p={p:.3f})")

if secondary_rejected:
    print(f"\nSecondary-pass rejections ({len(secondary_rejected)}):")
    for f, p, harmed in secondary_rejected:
        names = ", ".join(c for c, _ in harmed)
        print(f"  '{f}' would have harmed: {names}")


### 5.4 Final Stage 1 fit

In [ ]:
X_s1 = sm.add_constant(X_log[final_features_s1])
res_s1 = sm.Logit(y_log, X_s1).fit(disp=False)
y_prob_s1 = res_s1.predict(X_s1)

print(res_s1.summary())
print(f"\nAUC:         {roc_auc_score(y_log, y_prob_s1):.3f}")
print(f"McFadden R²: {res_s1.prsquared:.3f}")
print(f"LR Chi²:     {res_s1.llr:.2f}  (p = {res_s1.llr_pvalue:.2e})")


### 5.5 Optimal classification threshold (Youden's J)

The default 0.5 threshold is arbitrary. Youden's J selects the threshold that maximizes sensitivity + specificity. Reported accuracy in Table 4 uses this threshold.


In [ ]:
fpr, tpr, thresholds = roc_curve(y_log, y_prob_s1)
j_idx = (tpr - fpr).argmax()
best_threshold = thresholds[j_idx]
print(f"Best threshold by Youden's J = {best_threshold:.3f}")

y_pred_s1 = (y_prob_s1 >= best_threshold).astype(int)
print(f"\nAt threshold {best_threshold:.2f}:")
print(f"  Accuracy:  {accuracy_score(y_log, y_pred_s1):.3f}")
print(f"  Precision: {precision_score(y_log, y_pred_s1):.3f}")
print(f"  Recall:    {recall_score(y_log, y_pred_s1):.3f}")
print(f"  F1:        {f1_score(y_log, y_pred_s1):.3f}")


### 5.6 VIF check (Stage 1)

In [ ]:
def vif_table(X, label):
    Xc = sm.add_constant(X)
    vif = pd.DataFrame({
        'feature': Xc.columns,
        'VIF': [variance_inflation_factor(Xc.values, i) for i in range(Xc.shape[1])],
    })
    vif = vif[vif['feature'] != 'const'].sort_values('VIF', ascending=False)
    print(f"{label}  (max VIF = {vif['VIF'].max():.2f})")
    print(vif.to_string(index=False))
    return vif

vif_s1 = vif_table(X_log[final_features_s1], "Stage 1 VIF")


## 6. Stage 2 — Lognormal Duration Regression

Models `ln(1 + duration)` conditional on shutdown. The full predictor pool is admissible because the shutdown decision has already been made.

Same selection procedure as Stage 1: LASSO → forward → secondary → cleanup.


### 6.1 LASSO ranking

In [ ]:
stage2_pool = list(X_reg.columns)

X_std_s2 = pd.DataFrame(
    StandardScaler().fit_transform(X_reg[stage2_pool]),
    columns=stage2_pool, index=X_reg.index,
)

lasso_s2 = LassoCV(
    cv=5, max_iter=20000, random_state=RNG, n_jobs=-1,
).fit(X_std_s2, y_reg)

ranking_s2 = pd.Series(lasso_s2.coef_, index=stage2_pool).abs().sort_values(ascending=False)
print(f"Stage 2 LASSO optimal alpha = {lasso_s2.alpha_:.5f}")
print(f"Non-zero LASSO: {(ranking_s2 > 0).sum()} / {len(ranking_s2)}\n")
print("Top 15:")
print(ranking_s2.head(15).round(4).to_string())


### 6.2 Forward selection (with monitoring)

In [ ]:
# ============================================================
# Stage 2 — Forward selection with monitoring
# ============================================================
print("="*70)
print("FORWARD PASS (LASSO-ranked order)")
print("="*70)

selected_s2 = []
forward_drops_s2 = []

for col in ranking_s2.index:
    selected_s2.append(col)
    res = sm.OLS(y_reg, sm.add_constant(X_reg[selected_s2])).fit()
    p_new = res.pvalues[col]

    if p_new >= 0.05:
        selected_s2.remove(col)
        print(f"  ❌ {col:<70} p={p_new:.3f}")
        continue

    print(f"  ✅ {col:<70} p={p_new:.3f}")

    drops = [(c, res.pvalues[c]) for c in selected_s2
             if c != col and res.pvalues[c] >= 0.05]
    for d, p_d in drops:
        print(f"     ⚠️  → '{d}' became insignificant (p={p_d:.3f}), dropping")
        forward_drops_s2.append((col, d, p_d))
        selected_s2.remove(d)

print(f"\n→ {len(selected_s2)} features kept after forward pass")

# ----- SECONDARY PASS -----
print("\n" + "="*70)
print("SECONDARY PASS")
print("="*70)

core_s2 = selected_s2.copy()
secondary_added_s2, secondary_rejected_s2 = [], []

for f in [x for x in stage2_pool if x not in core_s2]:
    trial = core_s2 + [f]
    res = sm.OLS(y_reg, sm.add_constant(X_reg[trial])).fit()
    p_new = res.pvalues.get(f, 1.0)
    if p_new >= 0.05:
        continue
    harmed = [(c, res.pvalues.get(c, 1.0)) for c in core_s2
              if res.pvalues.get(c, 1.0) >= 0.05]
    if not harmed:
        core_s2.append(f)
        secondary_added_s2.append((f, p_new))
        print(f"  ✅ {f:<70} p={p_new:.3f}  (added)")
    else:
        harmed_str = ", ".join(f"{c}(p={p:.3f})" for c, p in harmed)
        secondary_rejected_s2.append((f, p_new, harmed))
        print(f"  ⚠️  {f:<70} p={p_new:.3f}  REJECTED — would harm: {harmed_str}")

if not secondary_added_s2 and not secondary_rejected_s2:
    print("  (none — no pool features become significant on top of core)")

# ----- FINAL CLEANUP -----
print("\n" + "="*70)
print("FINAL CLEANUP")
print("="*70)

final_features_s2 = core_s2
for it in range(5):
    res = sm.OLS(y_reg, sm.add_constant(X_reg[final_features_s2])).fit()
    nonsig = [(f, res.pvalues.get(f, 1.0)) for f in final_features_s2
              if res.pvalues.get(f, 1.0) >= 0.05]
    if not nonsig:
        if it == 0:
            print("  (no cleanup needed — all features significant)")
        break
    print(f"  Iteration {it+1}: dropping")
    for f, p in nonsig:
        print(f"     • {f}  p={p:.3f}")
    final_features_s2 = [f for f in final_features_s2 if f not in {n[0] for n in nonsig}]

# ----- SUMMARY -----
print("\n" + "="*70)
print("STAGE 2 SELECTION SUMMARY")
print("="*70)
print(f"Final features: {len(final_features_s2)}")
for f in final_features_s2:
    print(f"  • {f}")

if forward_drops_s2:
    print(f"\nForward-pass interaction drops ({len(forward_drops_s2)}):")
    for added, dropped, p in forward_drops_s2:
        print(f"  '{added}' bumped out '{dropped}' (p={p:.3f})")

if secondary_rejected_s2:
    print(f"\nSecondary-pass rejections ({len(secondary_rejected_s2)}):")
    for f, p, harmed in secondary_rejected_s2:
        names = ", ".join(c for c, _ in harmed)
        print(f"  '{f}' would have harmed: {names}")


In [ ]:
# ============================================================
# Stage 2 — final feature set
# ============================================================
# The forward-selection procedure above produces a ranked candidate set.
# The published Stage 2 model uses the list below, which is the
# forward-selection output with ACCIDENT_PSIG removed.
#
# ACCIDENT_PSIG is excluded from Stage 2 because (a) it is a Stage 1
# predictor (operating pressure at the time of incident shapes the
# shutdown decision rather than the recovery duration) and (b) it
# correlates with property damage on the log scale, so leaving it in
# Stage 2 mainly absorbs variance already explained by PRPTY. Pressure-
# related effects on duration are instead captured by the
# ACCIDENT_PRESSURE_PRESSURE EXCEEDED 110% OF MOP indicator, which
# corresponds to a distinct regulatory trigger under 49 CFR 192.933.

final_features_s2 = [
    'PRPTY',
    'INCIDENT_AREA_TYPE_UNDERGROUND',
    'COMMODITY_RELEASED_TYPE_HVL OR OTHER FLAMMABLE OR TOXIC FLUID WHICH IS A GAS AT AMBIENT CONDITIONS',
    'CAUSE_EQUIPMENT FAILURE',
    'CAUSE_INCORRECT OPERATION',
    'ACCIDENT_PRESSURE_PRESSURE EXCEEDED 110% OF MOP',
    'SYSTEM_PART_INVOLVED_ONSHORE TERMINAL/TANK FARM EQUIPMENT AND PIPING',
    'CROSSING_YES',
    'SYSTEM_PART_INVOLVED_ONSHORE PUMP/METER STATION EQUIPMENT AND PIPING',
    'PIPE_FACILITY_TYPE_INTRASTATE',
    'SCADA_IN_PLACE_IND_YES',
    'RELEASE_TYPE_OVERFILL OR OVERFLOW',
    'SYSTEM_PART_INVOLVED_OTHER',
    'SOIL_CONTAMINATION_YES',
    'RELEASE_TYPE_MECHANICAL PUNCTURE',
]

# Sanity: every feature must exist in X_reg
missing = [f for f in final_features_s2 if f not in X_reg.columns]
assert not missing, f"Missing from X_reg: {missing}"
print(f"Stage 2 features: {len(final_features_s2)}")
for f in final_features_s2:
    print(f"  - {f}")

# Fit on log-transformed duration
X_s2 = sm.add_constant(X_reg[final_features_s2])
res_s2 = sm.OLS(y_reg, X_s2).fit()
print(res_s2.summary())

# Prediction diagnostics on the log scale
y_pred_s2 = res_s2.predict(X_s2)
residuals_s2 = y_reg - y_pred_s2
mae  = np.mean(np.abs(residuals_s2))
rmse = np.sqrt(np.mean(residuals_s2 ** 2))
print(f"\nMAE  (log scale): {mae:.3f}")
print(f"RMSE (log scale): {rmse:.3f}")

# VIF check
vif_s2 = vif_table(X_reg[final_features_s2], 'Stage 2 VIF')

## 7. Descriptive Statistics

Tables for the paper:
- **Table 2**: categorical variables, grouped by pre-/post-decision
- **Table 3**: numerical variables (min / median / mean / max / std / skewness)


### 7.1 Numerical descriptive stats (Table 3)

In [ ]:
# Pull retained numerical features directly from the model variable lists
RETAINED_NUMERICAL = [
    f for f in (final_features_s1 + final_features_s2)
    if f in data_numerical.columns
]
# Deduplicate while preserving order
RETAINED_NUMERICAL = list(dict.fromkeys(RETAINED_NUMERICAL))

print(f"Retained numerical features ({len(RETAINED_NUMERICAL)}): {RETAINED_NUMERICAL}\n")

num_desc = pd.DataFrame({
    'Min':      data_numerical.loc[analytical_idx, RETAINED_NUMERICAL].min(),
    'Median':   data_numerical.loc[analytical_idx, RETAINED_NUMERICAL].median(),
    'Mean':     data_numerical.loc[analytical_idx, RETAINED_NUMERICAL].mean(),
    'Max':      data_numerical.loc[analytical_idx, RETAINED_NUMERICAL].max(),
    'Std.Dev':  data_numerical.loc[analytical_idx, RETAINED_NUMERICAL].std(),
    'Skewness': data_numerical.loc[analytical_idx, RETAINED_NUMERICAL].skew(),
}).round(2)

# Add Shutdown_Duration_Hours (response — nonzero shutdowns only, untrimmed)
dur_row = pd.DataFrame({
    'Min':      [duration_clean.min()],
    'Median':   [duration_clean.median()],
    'Mean':     [duration_clean.mean()],
    'Max':      [duration_clean.max()],
    'Std.Dev':  [duration_clean.std()],
    'Skewness': [duration_clean.skew()],
}, index=['Shutdown_Duration_Hours']).round(2)

num_desc = pd.concat([dur_row, num_desc])

# Tag pre/post for the paper grouping
def tag(name):
    if name == 'Shutdown_Duration_Hours': return 'response'
    return PREDECISION_VARS.get(name, 'pre')
num_desc['Type'] = [tag(c) for c in num_desc.index]

print(num_desc.to_string())
num_desc.to_csv(os.path.join(OUT_PATH, 'table3_numerical_descriptives.csv'))

### 7.2 Categorical descriptive stats (Table 2)

In [ ]:
# Identify categorical variables that have at least one retained dummy in either final model
all_final = set(final_features_s1) | set(final_features_s2)

retained_cats = []
for col in categorical_columns_list:
    has_retained = any(f.startswith(f"{col}_") for f in all_final)
    if has_retained:
        retained_cats.append(col)

print(f"Categorical variables retained in either stage ({len(retained_cats)}):")
for c in retained_cats:
    print(f"  • {c}")
print()

# Find reference category for each retained variable
def get_reference_category(col):
    all_cats = sorted(data_categorical.loc[analytical_idx, col].dropna().unique())
    dummy_cats = [f.replace(f"{col}_", "") for f in X_log.columns if f.startswith(f"{col}_")]
    reference = [c for c in all_cats if str(c) not in dummy_cats]
    return reference[0] if reference else None

# Build the table
rows = []
for col in retained_cats:
    s = data_categorical.loc[analytical_idx, col]
    counts = s.value_counts(dropna=False, normalize=True) * 100
    ref = get_reference_category(col)
    for cat, pct in counts.items():
        is_ref = (str(cat) == str(ref))
        rows.append({
            'Variable':    col,
            'Subcategory': str(cat) + (' *' if is_ref else ''),
            'Percentage':  round(pct, 2),
            'Type':        PREDECISION_VARS.get(col, 'pre'),
        })

# Add the response variable
y_dist = data.loc[analytical_idx, 'SHUTDOWN_DUE_ACCIDENT_IND'].value_counts(normalize=True) * 100
ref_y = 'NO'  # alphabetically first; explicitly set since response isn't dummified the same way
for cat, pct in y_dist.items():
    is_ref = (cat == ref_y)
    rows.append({
        'Variable':    'SHUTDOWN_DUE_ACCIDENT_IND',
        'Subcategory': cat + (' *' if is_ref else ''),
        'Percentage':  round(pct, 2),
        'Type':        'response',
    })

cat_desc = pd.DataFrame(rows)
print(cat_desc.to_string(index=False))
cat_desc.to_csv(os.path.join(OUT_PATH, 'table2_categorical_descriptives.csv'), index=False)

## 8. Hurdle Model Coefficient Table (Table 5)

Combines Stage 1 and Stage 2 estimates into one table, with variable descriptions pulled from the feature dictionary.


In [ ]:
def fit_table(res, prefix):
    return pd.DataFrame({
        f'{prefix}_Coef': res.params.round(3),
        f'{prefix}_Stat': res.tvalues.round(3),
        f'{prefix}_p':    res.pvalues.round(4),
    })

logit_df = fit_table(res_s1, 'Logit')
ols_df   = fit_table(res_s2, 'OLS')

# Union of indices, ordered: shared first, then logit-only, then ols-only
union = logit_df.index.union(ols_df.index)
hurdle = pd.concat([logit_df, ols_df], axis=1).reindex(union).fillna('--')

both       = hurdle.index.isin(set(logit_df.index) & set(ols_df.index))
logit_only = hurdle.index.isin(set(logit_df.index) - set(ols_df.index))
ols_only   = hurdle.index.isin(set(ols_df.index) - set(logit_df.index))

hurdle = pd.concat([hurdle[both], hurdle[logit_only], hurdle[ols_only]])

# Map column names to human-readable descriptions
fmap = dict(zip(feature_map['Feature Name'], feature_map['Description']))
hurdle.insert(0, 'Description', [fmap.get(i, i) for i in hurdle.index])

print(hurdle.to_string())
hurdle.to_csv(os.path.join(OUT_PATH, 'table5_hurdle_coefficients.csv'))


### 8.1 Model fit summary (Table 4)

In [ ]:
# Accuracy uses the Youden-J optimal threshold
fit_summary = pd.DataFrame({
    'Logistic': {
        'Log-Likelihood (Model)': round(res_s1.llf, 2),
        'McFadden R²':            round(res_s1.prsquared, 3),
        'LR Chi²':                round(res_s1.llr, 2),
        'LR p-value':             f"{res_s1.llr_pvalue:.2e}",
        'Number of Parameters':   int(res_s1.df_model) + 1,
        'Number of Observations': int(res_s1.nobs),
        'Accuracy':               round(accuracy_score(y_log, (y_prob_s1 >= best_threshold).astype(int)), 3),
        'ROC AUC':                round(roc_auc_score(y_log, y_prob_s1), 3),
    },
    'Lognormal': {
        'Log-Likelihood (Model)': round(res_s2.llf, 2),
        'Adjusted R²':            round(res_s2.rsquared_adj, 3),
        'F-statistic':            round(res_s2.fvalue, 2),
        'F-test p-value':         f"{res_s2.f_pvalue:.2e}",
        'Number of Parameters':   int(res_s2.df_model) + 1,
        'Number of Observations': int(res_s2.nobs),
        'MAE (log scale)':        round(np.mean(np.abs(residuals_s2)), 2),
        'RMSE (log scale)':       round(np.sqrt(np.mean(residuals_s2 ** 2)), 2),
    },
}).fillna('--')

print(fit_summary.to_string())
fit_summary.to_csv(os.path.join(OUT_PATH, 'table4_model_fit.csv'))


## 9. Figures

### 9.1 Shutdown duration distribution — raw and log-transformed (Figures 1 and 2)

In [ ]:
# ============================================================
# Figure 1 — raw shutdown duration: histogram + Q-Q plot
# ============================================================
fig1, axes1 = plt.subplots(1, 2, figsize=(12, 4.5), dpi=600)

# (a) Histogram
n_long  = (y_reg_raw > 5000).sum()
max_dur = y_reg_raw.max()

axes1[0].hist(y_reg_raw, bins=80, color='blue', edgecolor='black')
axes1[0].set_xlabel('Observed Shutdown Duration (hours)')
axes1[0].set_ylabel('Frequency')
axes1[0].annotate(
    f'{n_long} incidents > 5000 h\n(max = {max_dur:,.0f} h)',
    xy=(0.45, 0.55), xycoords='axes fraction',
    fontsize=9, ha='left', fontproperties=tnr,
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
              edgecolor='gray', alpha=0.9),
)

# (b) Q-Q plot
(osm, osr), (slope, intercept, r) = stats.probplot(y_reg_raw, dist='norm', plot=None)
axes1[1].scatter(osm, osr, s=8, color='blue', alpha=0.7, edgecolor='none')
axes1[1].plot(osm, slope * osm + intercept, color='darkred', lw=1.5)
axes1[1].set_xlabel('Theoretical quantiles')
axes1[1].set_ylabel('Observed Shutdown Duration (hours)')

for ax in axes1:
    ax.xaxis.set_major_formatter(FuncFormatter(comma_if_5plus))
    ax.yaxis.set_major_formatter(FuncFormatter(comma_if_5plus))
    apply_tnr(ax)

plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'fig1_duration_raw.pdf'),
            bbox_inches='tight', dpi=600)
plt.show()


# ============================================================
# Figure 2 — log-transformed shutdown duration: histogram + Q-Q plot
# ============================================================
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4.5), dpi=600)

# (a) Histogram of ln(duration)
axes2[0].hist(y_reg, bins=40, density=True, color='steelblue', edgecolor='black', alpha=0.7)
axes2[0].axvline(y_reg.mean(),   color='red',  ls='--', lw=2, label=f'Mean = {y_reg.mean():.2f}')
axes2[0].axvline(y_reg.median(), color='blue', ls='-.', lw=2, label=f'Median = {y_reg.median():.2f}')
axes2[0].legend()
axes2[0].set_xlabel('ln(Shutdown Duration [hours])')
axes2[0].set_ylabel('Density')

# (b) Q-Q plot of ln(duration)
(osm2, osr2), (slope2, intercept2, r2) = stats.probplot(y_reg, dist='norm', plot=None)
axes2[1].scatter(osm2, osr2, s=8, color='blue', alpha=0.7, edgecolor='none')
axes2[1].plot(osm2, slope2 * osm2 + intercept2, color='darkred', lw=1.5)
axes2[1].set_xlabel('Theoretical quantiles')
axes2[1].set_ylabel('ln(Shutdown Duration [hours])')

for ax in axes2:
    apply_tnr(ax)

plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'fig2_duration_log.pdf'),
            bbox_inches='tight', dpi=600)
plt.show()

print(f"Skewness raw: {stats.skew(y_reg_raw):.2f}")
print(f"Skewness log: {stats.skew(y_reg):.2f}")

### 9.2 Pearson correlation matrix (Figure 3)

Pairwise correlations among continuous variables retained in the final hurdle model, all on the natural-log scale.

In [ ]:
# Correlation matrix of continuous variables retained in the final hurdle model
# (matches Figure 3 in the paper).

duration_for_corr = duration_clean.loc[X_reg.index]
d = data.loc[X_reg.index, ['PRPTY', 'ACCIDENT_PSIG']].copy()
d['Shutdown_Duration_Hours'] = duration_for_corr

ld = pd.DataFrame({
    'ln(Shutdown Duration [h])':            np.log(d['Shutdown_Duration_Hours']),
    'ln(1 + Property Damage [$])':          np.log1p(d['PRPTY']),
    'ln(1 + Accident Pressure [psig])':     np.log1p(d['ACCIDENT_PSIG']),
})

corr = ld.corr(method='pearson')

fig, ax = plt.subplots(figsize=(7.5, 6.5), dpi=600)
hm = sns.heatmap(
    corr, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
    annot_kws={'size': 12, 'weight': 'bold', 'fontproperties': tnr},
    ax=ax,
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
ax.tick_params(left=False, bottom=False)
apply_tnr(ax)

cbar = hm.collections[0].colorbar
cbar.ax.yaxis.label.set_fontproperties(tnr)
for lbl in cbar.ax.get_yticklabels():
    lbl.set_fontproperties(tnr)

plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'fig3_correlation_matrix.pdf'),
            bbox_inches='tight', dpi=600)
plt.show()
print(f"n = {len(ld)} shutdown incidents")

### 9.3 Stage 2 residuals and observed vs predicted (Figure 4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=600)

# (a) Residuals vs predicted
axes[0].scatter(y_pred_s2, residuals_s2, alpha=0.25, s=15, color='blue', edgecolor='none')
axes[0].axhline(0, color='red', ls='--', lw=1)
axes[0].set_xlabel('Predicted ln(Shutdown Duration [hours])')
axes[0].set_ylabel('Residuals (Observed − Predicted)')

# Make residual y-axis symmetric around 0
res_lim = max(abs(residuals_s2.min()), abs(residuals_s2.max()))
axes[0].set_ylim(-res_lim, res_lim)

# (b) Observed vs predicted
axes[1].scatter(y_reg, y_pred_s2, alpha=0.25, s=15, color='blue', edgecolor='none')
lim = [min(y_reg.min(), y_pred_s2.min()), max(y_reg.max(), y_pred_s2.max())]
axes[1].plot(lim, lim, color='red', ls='--', lw=1)
axes[1].set_xlabel('ln(Shutdown Duration [hours])')
axes[1].set_ylabel('Predicted ln(Shutdown Duration [hours])')

for ax in axes:
    apply_tnr(ax)

plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'fig4_residuals.pdf'),
            bbox_inches='tight', dpi=600)
plt.show()

## 10. Supplementary Robustness Checks

These analyses are reported in the paper's supplementary material (Tables S1–S5, Figure S1) and §5.3 of the main text.

- **10.1** Year fixed effects (Tables S3–S4)
- **10.2** Heavy-tail diagnosis: lognormal vs Burr12 fit, GPD upper-tail (Figure S1, Table S5)
- **10.3** MICE multiple imputation — BayesianRidge (Tables S1–S2, BR columns)
- **10.4** MICE multiple imputation — PMM (Tables S1–S2, PMM columns)

### 10.1 Year fixed effects

In [ ]:
year_s1 = data.loc[X_log.index, 'IYEAR'].astype(int)
year_s2 = data.loc[X_reg.index, 'IYEAR'].astype(int)

# Year dummies (reference = 2015)
yd_s1 = pd.get_dummies(year_s1, prefix='YEAR').astype(int)
yd_s2 = pd.get_dummies(year_s2, prefix='YEAR').astype(int)
if 'YEAR_2015' in yd_s1.columns: yd_s1 = yd_s1.drop(columns='YEAR_2015')
if 'YEAR_2015' in yd_s2.columns: yd_s2 = yd_s2.drop(columns='YEAR_2015')

# Stage 1 with year FE
X_s1_FE = sm.add_constant(pd.concat([X_log[final_features_s1], yd_s1], axis=1))
res_s1_FE = sm.Logit(y_log, X_s1_FE).fit(disp=False)
LR_s1   = 2 * (res_s1_FE.llf - res_s1.llf)
p_LR_s1 = chi2.sf(LR_s1, yd_s1.shape[1])

# Stage 2 with year FE
X_s2_FE = sm.add_constant(pd.concat([X_reg[final_features_s2], yd_s2], axis=1))
res_s2_FE = sm.OLS(y_reg, X_s2_FE).fit()
F_s2 = res_s2_FE.f_test(', '.join([f'{c} = 0' for c in yd_s2.columns]))

print("="*70)
print("YEAR FE ROBUSTNESS")
print("="*70)
print(f"Stage 1 (logit):")
print(f"  No FE:   LL = {res_s1.llf:.2f}, McF R² = {res_s1.prsquared:.3f}, "
      f"AUC = {roc_auc_score(y_log, res_s1.predict(X_s1)):.3f}")
print(f"  With FE: LL = {res_s1_FE.llf:.2f}, McF R² = {res_s1_FE.prsquared:.3f}, "
      f"AUC = {roc_auc_score(y_log, res_s1_FE.predict(X_s1_FE)):.3f}")
print(f"  LR test: chi² = {LR_s1:.2f}, df = {yd_s1.shape[1]}, p = {p_LR_s1:.4f}")

print(f"\nStage 2 (OLS):")
print(f"  No FE:   Adj.R² = {res_s2.rsquared_adj:.3f}, AIC = {res_s2.aic:.1f}")
print(f"  With FE: Adj.R² = {res_s2_FE.rsquared_adj:.3f}, AIC = {res_s2_FE.aic:.1f}")
print(f"  F test:  F = {float(F_s2.fvalue):.2f}, df = ({int(F_s2.df_num)}, {int(F_s2.df_denom)}), "
      f"p = {float(F_s2.pvalue):.4f}")


In [ ]:
# Coefficient stability tables
def stability_table(no_fe, with_fe, feats, label):
    df = pd.DataFrame({
        'No FE Coef':   no_fe.params.loc[feats].round(3),
        'With FE Coef': with_fe.params.loc[feats].round(3),
        'No FE p':      no_fe.pvalues.loc[feats].round(4),
        'With FE p':    with_fe.pvalues.loc[feats].round(4),
    })
    df['Δ%']  = ((df['With FE Coef'] - df['No FE Coef']) / df['No FE Coef'].abs() * 100).round(1)
    df['flip'] = np.sign(df['No FE Coef']) != np.sign(df['With FE Coef'])
    df['sig_change'] = (df['No FE p'] < 0.05) != (df['With FE p'] < 0.05)
    print(f"\n{label} stability:")
    print(df.to_string())
    print(f"Max |Δ%| = {df['Δ%'].abs().max():.1f}%, flips = {int(df['flip'].sum())}, "
          f"sig changes = {int(df['sig_change'].sum())}")
    return df

stab_s1 = stability_table(res_s1, res_s1_FE, final_features_s1, "Stage 1")
stab_s2 = stability_table(res_s2, res_s2_FE, final_features_s2, "Stage 2")

stab_s1.to_csv(os.path.join(OUT_PATH, 'tableS3_yearFE_stage1.csv'))
stab_s2.to_csv(os.path.join(OUT_PATH, 'tableS4_yearFE_stage2.csv'))


### 10.2 Heavy-tail diagnosis: lognormal vs Burr12, GPD upper tail

Compares the lognormal fit to a heavier-tailed Burr12 by AIC, and fits a Generalized Pareto distribution to the upper tail at varying threshold quantiles. Q-Q plots provide visual fit assessment.

Reported in the paper as Figure S1 and Table S5 (supplementary material). The trimmed-sample sensitivity below is internal-only and is not reported in the paper.

In [ ]:
# ============================================================
# Distribution fits: Lognormal vs Burr12, plus GPD on the tail
# ============================================================
from scipy.stats import burr12, lognorm, genpareto, kstest

dur = duration_clean[duration_clean > 0].values
print("="*70)
print(f"DISTRIBUTION FITS  (n = {len(dur)} nonzero durations)")
print("="*70)

# --- Lognormal ---
ln_s, _, ln_scale = lognorm.fit(dur, floc=0)
ln_loglik = lognorm.logpdf(dur, ln_s, 0, ln_scale).sum()
ln_aic = -2 * ln_loglik + 2 * 2
ln_bic = -2 * ln_loglik + np.log(len(dur)) * 2
ln_ks  = kstest(dur, 'lognorm', args=(ln_s, 0, ln_scale)).statistic

# --- Burr12 ---
br_c, br_d, _, br_scale = burr12.fit(dur, floc=0)
br_loglik = burr12.logpdf(dur, br_c, br_d, 0, br_scale).sum()
br_aic = -2 * br_loglik + 2 * 3
br_bic = -2 * br_loglik + np.log(len(dur)) * 3
br_ks  = kstest(dur, 'burr12', args=(br_c, br_d, 0, br_scale)).statistic

print(f"\nLognormal: s={ln_s:.3f}, scale={ln_scale:.1f}")
print(f"  log-lik={ln_loglik:.1f}, AIC={ln_aic:.1f}, BIC={ln_bic:.1f}, KS={ln_ks:.4f}")
print(f"\nBurr12:    c={br_c:.3f}, d={br_d:.3f}, scale={br_scale:.1f}")
print(f"  log-lik={br_loglik:.1f}, AIC={br_aic:.1f}, BIC={br_bic:.1f}, KS={br_ks:.4f}")

# Verdict
winner = min([('Lognormal', ln_aic), ('Burr12', br_aic)], key=lambda x: x[1])
print(f"\nLower AIC wins: {winner[0]} (AIC = {winner[1]:.1f})")
print(f"AIC difference (Burr − Lognormal): {br_aic - ln_aic:+.1f}")
print(f"  • |ΔAIC| < 2  →  models indistinguishable")
print(f"  • |ΔAIC| > 10 →  decisive")

# --- GPD over varying thresholds ---
print(f"\n{'='*70}")
print("Generalized Pareto over varying thresholds")
print("ξ > 0 ⇒ heavy tail (power-law); ξ ≈ 0 ⇒ exponential; ξ < 0 ⇒ bounded")
print("="*70)
gpd_rows = []
for q in [0.90, 0.95, 0.97, 0.99]:
    thr = np.quantile(dur, q)
    excess = dur[dur > thr] - thr
    if len(excess) < 10: continue
    xi, _, sigma = genpareto.fit(excess, floc=0)
    print(f"  q={q:.2f}, thr={thr:>7.0f}h, n_excess={len(excess):4d}, "
          f"ξ={xi:>+6.3f}, σ={sigma:>7.1f}")
    gpd_rows.append({'quantile': q, 'threshold_h': thr, 'n_excess': len(excess),
                     'xi': xi, 'sigma': sigma})

pd.DataFrame(gpd_rows).to_csv(os.path.join(OUT_PATH, 'tableS5_GPD_fits.csv'), index=False)


In [ ]:
# ============================================================
# Q-Q plots: Lognormal vs Burr12
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), dpi=600)

n = len(dur)
emp_q = np.sort(dur)
plot_pos = (np.arange(1, n + 1) - 0.5) / n   # Hazen plotting positions

# Lognormal Q-Q
ln_theo = lognorm.ppf(plot_pos, ln_s, 0, ln_scale)
axes[0].scatter(ln_theo, emp_q, s=8, alpha=0.4, color='blue', edgecolor='none')
lim = [min(ln_theo.min(), emp_q.min()), max(ln_theo.max(), emp_q.max())]
axes[0].plot(lim, lim, 'r--', lw=1.5, label='45° reference')
axes[0].set_xscale('log'); axes[0].set_yscale('log')
axes[0].set_xlabel('Lognormal theoretical quantiles (log scale)')
axes[0].set_ylabel('Observed duration [hours] (log scale)')
axes[0].set_title(f'(a) Q-Q vs Lognormal\nKS = {ln_ks:.3f}')
axes[0].legend()

# Burr12 Q-Q
br_theo = burr12.ppf(plot_pos, br_c, br_d, 0, br_scale)
axes[1].scatter(br_theo, emp_q, s=8, alpha=0.4, color='blue', edgecolor='none')
lim = [min(br_theo.min(), emp_q.min()), max(br_theo.max(), emp_q.max())]
axes[1].plot(lim, lim, 'r--', lw=1.5, label='45° reference')
axes[1].set_xscale('log'); axes[1].set_yscale('log')
axes[1].set_xlabel('Burr12 theoretical quantiles (log scale)')
axes[1].set_ylabel('Observed duration [hours] (log scale)')
axes[1].set_title(f'(b) Q-Q vs Burr12\nKS = {br_ks:.3f}')
axes[1].legend()

for ax in axes:
    apply_tnr(ax)

plt.tight_layout()
plt.savefig(os.path.join(OUT_PATH, 'figS1_qq_lognormal_vs_burr12.pdf'),
            bbox_inches='tight', dpi=600)
plt.show()

### 10.3 MICE multiple imputation — BayesianRidge

Refits both stages on `m=10` multiply-imputed datasets. Pools coefficients with **Rubin's rules**: pooled estimate is the mean of within-imputation estimates; pooled variance is within + (1 + 1/m) · between.

Compares complete-case (CC) vs MICE-pooled coefficients to check whether the complete-case analysis introduces meaningful bias. Reported in the paper's supplementary material as Tables S1 (Stage 1) and S2 (Stage 2), under the BR columns.

#### 10.3.1 Build imputation matrix

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer

# Restrict to rows where the outcome is observed (only filter)
shutdown_known = data['SHUTDOWN_DUE_ACCIDENT_IND'].notna()
imp_base = data.loc[shutdown_known].copy()
print(f"Shutdown-known sample (full, before complete-case filter): {len(imp_base)}")

def onehot_preserve_nan(df_cat):
    """One-hot encode each column; rows with NaN get NaN in ALL dummies
    of that column (so MICE imputes them jointly)."""
    out = []
    for col in df_cat.columns:
        s = df_cat[col]
        nan_mask = s.isna()
        dummies = pd.get_dummies(s, prefix=col, drop_first=True).astype(float)
        dummies.loc[nan_mask, :] = np.nan
        out.append(dummies)
    return pd.concat(out, axis=1)

# Encode predictors EXACTLY like the CC pipeline (log1p numericals + dummies)
cat_encoded = onehot_preserve_nan(imp_base[categorical_columns_list])
num_logged  = np.log1p(imp_base[numeric_columns_list])

# Track column types so we can round dummies back after imputation
dummy_cols = list(cat_encoded.columns)
num_cols   = list(num_logged.columns)

# Outcome dummy (this is the response, not a predictor — drop from impute matrix)
sd_col = 'SHUTDOWN_DUE_ACCIDENT_IND_YES'
if sd_col in cat_encoded.columns:
    cat_encoded = cat_encoded.drop(columns=sd_col)
    dummy_cols.remove(sd_col)

# Outcome (fully observed by construction)
y_outcome = (imp_base['SHUTDOWN_DUE_ACCIDENT_IND'] == 'YES').astype(int)

# Imputation matrix: predictors + outcome (outcome informs imputation but won't be modified)
imp_matrix = pd.concat([cat_encoded, num_logged], axis=1)
imp_matrix['_outcome'] = y_outcome.values

print(f"Imputation matrix: {imp_matrix.shape}")
print(f"  Dummy columns:   {len(dummy_cols)}")
print(f"  Numeric columns: {len(num_cols)}")
print(f"  Missing cells:   {imp_matrix.isna().sum().sum()}")
print(f"  Missing rate:    {imp_matrix.isna().sum().sum() / imp_matrix.size:.3%}")

# Sanity check
missing_s1 = [f for f in final_features_s1 if f not in imp_matrix.columns]
missing_s2 = [f for f in final_features_s2 if f not in imp_matrix.columns]
assert not missing_s1, f"Missing from impute matrix: {missing_s1}"
assert not missing_s2, f"Missing from impute matrix: {missing_s2}"
print(f"\n✅ All Stage 1 ({len(final_features_s1)}) and Stage 2 ({len(final_features_s2)}) features present.")


#### 10.3.2 Pre-imputation diagnostic

In [ ]:
features_with_nulls = imp_matrix.columns[imp_matrix.isna().any()].tolist()
features_with_nulls = [f for f in features_with_nulls if f != '_outcome']

print(f"Features with at least one null: {len(features_with_nulls)}\n")

pre_diag_rows = []
for f in features_with_nulls:
    s = imp_matrix[f]
    n_null = s.isna().sum()
    pct_null = n_null / len(s) * 100
    is_dummy = f in dummy_cols
    pre_diag_rows.append({
        'feature':  f,
        'type':     'dummy' if is_dummy else 'numeric (log1p)',
        'n_null':   int(n_null),
        '%_null':   round(pct_null, 1),
        'mean':     round(s.mean(), 3),
        'median':   round(s.median(), 3),
        'std':      round(s.std(), 3),
        'min':      round(s.min(), 3),
        'max':      round(s.max(), 3),
    })

pre_diag = pd.DataFrame(pre_diag_rows).sort_values('%_null', ascending=False)
print(pre_diag.to_string(index=False))
pre_diag.to_csv(os.path.join(OUT_PATH, 'mice_prediag.csv'), index=False)


#### 10.3.3 Run BayesianRidge MICE (m=10)

In [ ]:
M = 10
imputed = []

for i in range(M):
    imp = IterativeImputer(
        estimator=BayesianRidge(),
        max_iter=10,
        random_state=i,
        sample_posterior=True,
    )
    arr = imp.fit_transform(imp_matrix)
    df_imp = pd.DataFrame(arr, columns=imp_matrix.columns, index=imp_matrix.index)

    # Round dummies to {0,1}; clip log1p numerics to ≥0; restore outcome exactly
    df_imp[dummy_cols] = df_imp[dummy_cols].clip(0, 1).round().astype(int)
    df_imp[num_cols]   = df_imp[num_cols].clip(lower=0)
    df_imp['_outcome'] = y_outcome.values

    imputed.append(df_imp)
    print(f"  Imputation {i+1}/{M} done")

print(f"\n✅ {M} BayesianRidge imputations generated")


#### 10.3.4 Post-imputation diagnostic

In [ ]:
post_rows = []
for f in features_with_nulls:
    is_dummy = f in dummy_cols
    pre_s    = imp_matrix[f]
    n_null   = int(pre_s.isna().sum())
    pre_mean = pre_s.mean()
    pre_std  = pre_s.std()

    means_imp = [df[f].mean() for df in imputed]
    stds_imp  = [df[f].std()  for df in imputed]
    null_mask = pre_s.isna()
    imp_only_mean = [df.loc[null_mask, f].mean() for df in imputed]

    post_rows.append({
        'feature':         f,
        'type':            'dummy' if is_dummy else 'numeric (log1p)',
        'n_null':          n_null,
        'pre_mean (obs)':  round(pre_mean, 3),
        'post_mean (all)': round(np.mean(means_imp), 3),
        'imp_cells_mean':  round(np.mean(imp_only_mean), 3),
        'pre_std (obs)':   round(pre_std, 3),
        'post_std (all)':  round(np.mean(stds_imp), 3),
        'mean_Δ%':         round((np.mean(means_imp) - pre_mean) / abs(pre_mean) * 100, 1)
                            if pre_mean != 0 else np.nan,
    })

post_diag = pd.DataFrame(post_rows).sort_values('feature')
print(f"Post-imputation diagnostic (averaged across m={M} imputed datasets):\n")
print(post_diag.to_string(index=False))
post_diag.to_csv(os.path.join(OUT_PATH, 'mice_postdiag.csv'), index=False)

# Sanity flags
print("\n" + "="*70)
print("SANITY CHECKS")
print("="*70)
bad_dummies = []
for f in features_with_nulls:
    if f in dummy_cols:
        for df in imputed:
            vals = set(df[f].unique())
            if not vals.issubset({0, 1}):
                bad_dummies.append((f, vals)); break
if bad_dummies:
    print(f"⚠️ Dummies not in {{0,1}}: {bad_dummies}")
else:
    print(f"✅ All {sum(1 for f in features_with_nulls if f in dummy_cols)} dummies are clean {{0,1}}")

big_shift = post_diag[post_diag['mean_Δ%'].abs() > 50]
if len(big_shift):
    print(f"\n⚠️ Large mean shifts (> 50%):")
    print(big_shift[['feature', 'pre_mean (obs)', 'post_mean (all)', 'mean_Δ%']].to_string(index=False))
else:
    print("✅ No feature shifted by > 50% in mean")


#### 10.3.5 Refit both stages, pool with Rubin's rules

In [ ]:
def pool(results):
    """Rubin's rules: pooled estimate = mean of within-imputation estimates;
    pooled variance = within + (1+1/m) · between."""
    coefs = np.array([r.params.values for r in results])
    vars_ = np.array([(r.bse.values) ** 2 for r in results])
    pooled_b = coefs.mean(0)
    within   = vars_.mean(0)
    between  = coefs.var(0, ddof=1)
    pooled_v = within + (1 + 1/len(results)) * between
    se = np.sqrt(pooled_v)
    z  = pooled_b / se
    p  = 2 * (1 - norm.cdf(np.abs(z)))
    return pd.DataFrame({
        'pooled_coef': pooled_b, 'pooled_se': se, 'pooled_z': z, 'pooled_p': p,
    }, index=results[0].params.index)

# --- Stage 1 ---
s1_results = []
for idf in imputed:
    Xi = idf[final_features_s1]
    yi = idf['_outcome']
    r = sm.Logit(yi, sm.add_constant(Xi)).fit(disp=False, maxiter=200)
    s1_results.append(r)
pooled_s1 = pool(s1_results)

cmp_s1 = pd.DataFrame({
    'CC_coef':   res_s1.params.round(3),
    'CC_p':      res_s1.pvalues.round(4),
    'MICE_coef': pooled_s1['pooled_coef'].reindex(res_s1.params.index).round(3),
    'MICE_p':    pooled_s1['pooled_p'].reindex(res_s1.params.index).round(4),
})
cmp_s1['Δ']  = (cmp_s1['MICE_coef'] - cmp_s1['CC_coef']).round(3)
cmp_s1['Δ%'] = (cmp_s1['Δ'] / cmp_s1['CC_coef'].abs() * 100).round(1)
cmp_s1['sig_change'] = (cmp_s1['CC_p'] < 0.05) != (cmp_s1['MICE_p'] < 0.05)

print("="*70)
print("STAGE 1 — Complete-case vs MICE-BR pooled")
print("="*70)
print(cmp_s1.to_string())
print(f"\nMax |Δ%|: {cmp_s1['Δ%'].abs().max():.1f}%")
print(f"Significance changes: {int(cmp_s1['sig_change'].sum())}")
cmp_s1.to_csv(os.path.join(OUT_PATH, 'tableS1_MICE_BR_stage1.csv'))

# --- Stage 2 ---
duration = imp_base['Shutdown_Duration_Hours']

s2_results = []
for idf in imputed:
    has_sd = idf['_outcome'] == 1
    yi_idx = has_sd[has_sd].index
    yi = np.log(duration.loc[yi_idx])      # log scale, untrimmed
    Xi = idf.loc[yi_idx, final_features_s2]
    r = sm.OLS(yi, sm.add_constant(Xi)).fit()
    s2_results.append(r)
pooled_s2 = pool(s2_results)

cmp_s2 = pd.DataFrame({
    'CC_coef':   res_s2.params.round(3),
    'CC_p':      res_s2.pvalues.round(4),
    'MICE_coef': pooled_s2['pooled_coef'].reindex(res_s2.params.index).round(3),
    'MICE_p':    pooled_s2['pooled_p'].reindex(res_s2.params.index).round(4),
})
cmp_s2['Δ']  = (cmp_s2['MICE_coef'] - cmp_s2['CC_coef']).round(3)
cmp_s2['Δ%'] = (cmp_s2['Δ'] / cmp_s2['CC_coef'].abs() * 100).round(1)
cmp_s2['sig_change'] = (cmp_s2['CC_p'] < 0.05) != (cmp_s2['MICE_p'] < 0.05)

print("\n" + "="*70)
print("STAGE 2 — Complete-case vs MICE-BR pooled")
print("="*70)
print(cmp_s2.to_string())
print(f"\nMax |Δ%|: {cmp_s2['Δ%'].abs().max():.1f}%")
print(f"Significance changes: {int(cmp_s2['sig_change'].sum())}")
cmp_s2.to_csv(os.path.join(OUT_PATH, 'tableS2_MICE_BR_stage2.csv'))


### 10.4 MICE multiple imputation — PMM

**Predictive Mean Matching** is a donor-based MICE variant: for each missing value, predict it from a regression, then donate one of the *k observed* rows whose predicted value is closest. PMM preserves the empirical distribution of observed values (zero-spikes, discrete clustering) which BayesianRidge tends to smooth over.

Agreement between BR and PMM indicates robustness to imputer choice. Reported in the paper's supplementary material as the PMM columns of Tables S1 and S2.

#### 10.4.1 PMM imputer + run

In [ ]:
class PMMImputer:
    """Predictive mean matching: predict the missing value, then donate
    the actual y from one of the k observed rows whose predicted value
    is closest. Preserves zero-spikes and discrete clustering that
    BayesianRidge smooths over."""

    def __init__(self, n_donors=5, max_iter=10, random_state=0):
        self.n_donors = n_donors
        self.max_iter = max_iter
        self.rng = np.random.default_rng(random_state)

    def fit_transform(self, X):
        X = X.copy()
        cols = X.columns.tolist()
        miss_mask = X.isna()

        # Initial fill
        for c in cols:
            if miss_mask[c].any():
                if X[c].dropna().nunique() <= 2:
                    fill = X[c].mode().iloc[0] if not X[c].dropna().empty else 0
                else:
                    fill = X[c].mean()
                X.loc[miss_mask[c], c] = fill

        # Iterative passes
        for it in range(self.max_iter):
            order = list(cols); self.rng.shuffle(order)

            for c in order:
                if not miss_mask[c].any(): continue
                predictors = [p for p in cols if p != c]
                X_pred = X[predictors].values
                y = X[c].values

                obs_idx  = np.where(~miss_mask[c].values)[0]
                miss_idx = np.where( miss_mask[c].values)[0]
                if len(obs_idx) < self.n_donors + 1: continue

                model = BayesianRidge()
                model.fit(X_pred[obs_idx], y[obs_idx])
                yhat_obs  = model.predict(X_pred[obs_idx])
                yhat_miss = model.predict(X_pred[miss_idx])

                # For each missing row, donate from the k closest observed
                imputed_local = np.empty(len(miss_idx))
                for j, ym in enumerate(yhat_miss):
                    dists = np.abs(yhat_obs - ym)
                    donor_pool = np.argpartition(dists, self.n_donors)[:self.n_donors]
                    chosen = self.rng.choice(donor_pool)
                    imputed_local[j] = y[obs_idx[chosen]]

                X.iloc[miss_idx, X.columns.get_loc(c)] = imputed_local

        return X


imputed_pmm = []
print(f"Running PMM with m={M} (slower than BayesianRidge — a few minutes)...")
for i in range(M):
    pmm = PMMImputer(n_donors=5, max_iter=10, random_state=i)
    df_imp = pmm.fit_transform(imp_matrix)

    # Defensive post-processing — PMM donates real values, so dummies should
    # already be {0,1}; round just in case
    df_imp[dummy_cols] = df_imp[dummy_cols].clip(0, 1).round().astype(int)
    df_imp[num_cols]   = df_imp[num_cols].clip(lower=0)
    df_imp['_outcome'] = y_outcome.values

    imputed_pmm.append(df_imp)
    print(f"  PMM imputation {i+1}/{M} done")

print(f"\n✅ {M} PMM imputations generated")


#### 10.4.2 Refit on PMM data, side-by-side CC / BR / PMM

In [ ]:
# --- Stage 1 ---
s1_pmm_results = []
for idf in imputed_pmm:
    Xi = idf[final_features_s1]
    yi = idf['_outcome']
    r = sm.Logit(yi, sm.add_constant(Xi)).fit(disp=False, maxiter=200)
    s1_pmm_results.append(r)
pooled_s1_pmm = pool(s1_pmm_results)

cmp_s1_pmm = pd.DataFrame({
    'CC_coef':   res_s1.params.round(3),
    'CC_p':      res_s1.pvalues.round(4),
    'BR_coef':   pooled_s1['pooled_coef'].reindex(res_s1.params.index).round(3),
    'BR_p':      pooled_s1['pooled_p'].reindex(res_s1.params.index).round(4),
    'PMM_coef':  pooled_s1_pmm['pooled_coef'].reindex(res_s1.params.index).round(3),
    'PMM_p':     pooled_s1_pmm['pooled_p'].reindex(res_s1.params.index).round(4),
})
cmp_s1_pmm['Δ_PMM_vs_CC%']  = ((cmp_s1_pmm['PMM_coef'] - cmp_s1_pmm['CC_coef'])
                               / cmp_s1_pmm['CC_coef'].abs() * 100).round(1)
cmp_s1_pmm['sig_change']    = (cmp_s1_pmm['CC_p'] < 0.05) != (cmp_s1_pmm['PMM_p'] < 0.05)

print("="*70)
print("STAGE 1 — CC vs BayesianRidge vs PMM")
print("="*70)
print(cmp_s1_pmm.to_string())
print(f"\nMax |Δ% vs CC|: {cmp_s1_pmm['Δ_PMM_vs_CC%'].abs().max():.1f}%")
print(f"Significance changes vs CC: {int(cmp_s1_pmm['sig_change'].sum())}")
cmp_s1_pmm.to_csv(os.path.join(OUT_PATH, 'tableS1_MICE_combined_stage1.csv'))

# --- Stage 2 ---
s2_pmm_results = []
for idf in imputed_pmm:
    has_sd = idf['_outcome'] == 1
    yi_idx = has_sd[has_sd].index
    yi = np.log(duration.loc[yi_idx])
    Xi = idf.loc[yi_idx, final_features_s2]
    r = sm.OLS(yi, sm.add_constant(Xi)).fit()
    s2_pmm_results.append(r)
pooled_s2_pmm = pool(s2_pmm_results)

cmp_s2_pmm = pd.DataFrame({
    'CC_coef':   res_s2.params.round(3),
    'CC_p':      res_s2.pvalues.round(4),
    'BR_coef':   pooled_s2['pooled_coef'].reindex(res_s2.params.index).round(3),
    'BR_p':      pooled_s2['pooled_p'].reindex(res_s2.params.index).round(4),
    'PMM_coef':  pooled_s2_pmm['pooled_coef'].reindex(res_s2.params.index).round(3),
    'PMM_p':     pooled_s2_pmm['pooled_p'].reindex(res_s2.params.index).round(4),
})
cmp_s2_pmm['Δ_PMM_vs_CC%']  = ((cmp_s2_pmm['PMM_coef'] - cmp_s2_pmm['CC_coef'])
                               / cmp_s2_pmm['CC_coef'].abs() * 100).round(1)
cmp_s2_pmm['sig_change']    = (cmp_s2_pmm['CC_p'] < 0.05) != (cmp_s2_pmm['PMM_p'] < 0.05)

print("\n" + "="*70)
print("STAGE 2 — CC vs BayesianRidge vs PMM")
print("="*70)
print(cmp_s2_pmm.to_string())
print(f"\nMax |Δ% vs CC|: {cmp_s2_pmm['Δ_PMM_vs_CC%'].abs().max():.1f}%")
print(f"Significance changes vs CC: {int(cmp_s2_pmm['sig_change'].sum())}")
cmp_s2_pmm.to_csv(os.path.join(OUT_PATH, 'tableS2_MICE_combined_stage2.csv'))


## 11. Save final model objects

In [ ]:
import pickle

artifacts = {
    'res_s1':            res_s1,
    'res_s2':            res_s2,
    'res_s1_FE':         res_s1_FE,
    'res_s2_FE':         res_s2_FE,
    'final_features_s1': final_features_s1,
    'final_features_s2': final_features_s2,
    'predecision_pool':  predecision_pool,
    'best_threshold':    best_threshold,
}
with open(os.path.join(OUT_PATH, 'final_models.pkl'), 'wb') as f:
    pickle.dump(artifacts, f)

print(f"Saved final models to {os.path.join(OUT_PATH, 'final_models.pkl')}")
print("\n✅ Notebook run complete. Outputs in:", OUT_PATH)
